In [1]:
import pandas as pd
import numpy as np
import os
import pickle
from typing import Dict, Any

# --- 모델 하이퍼파라미터 (규칙) 정의 ---

# 1. 서프라이즈 필터 (과제 1/2)
Z_SCORE_THRESHOLD = 2.0

# 2. GICS 필터 (과제 3: 훈련 기간 최적 Sector)
# (이전에 35.0이 최적이었음을 기억하고 해당 Sector에 집중)
MOST_SENSITIVE_GICS_SECTORS = [35.0] 

# 3. 모멘텀 필터 (PDF Page 5)
MOMENTUM_PERIOD = 20 # 20일 기준
MOMENTUM_THRESHOLD = 0.08 # 8% 이상 상승

# 4. 거래량 필터 (PDF Page 5)
VOLUME_RATIO_THRESHOLD = 2.0 # 2.0배 이상 급증

# 모델 메타데이터
MODEL_FILE_NAME = "model_ver3.pkl"
OUT_DIR = "../../output/model" 
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
def get_investment_decision_ver3(
    surprise_z: float, 
    gics_code: float, 
    momentum_rate: float, 
    volume_ratio: float
) -> Dict[str, Any]:
    """
    최종 통합 규칙 기반 모델 (Model Version 3).
    Z-Score, GICS, Momentum, Volume의 4가지 조건을 모두 충족할 때만 BUY 결정.
    """
    
    # 1. Decision Logic Check
    
    # Z-Score 조건 (강력한 Positive Signal)
    is_surprise_ok = (surprise_z > Z_SCORE_THRESHOLD)
    
    # GICS 조건 (최적 산업군에 속하는가?)
    is_gics_ok = (gics_code in MOST_SENSITIVE_GICS_SECTORS)
    
    # 모멘텀 조건 (20일 동안 8% 이상 상승했는가?)
    is_momentum_ok = (momentum_rate >= MOMENTUM_THRESHOLD)
    
    # 거래량 조건 (평소 대비 2배 이상 급증했는가?)
    is_volume_ok = (volume_ratio >= VOLUME_RATIO_THRESHOLD)
    
    
    # 2. Final Decision (ALL-IN)
    if is_surprise_ok and is_gics_ok and is_momentum_ok and is_volume_ok:
        decision = 'BUY'
        # 예상 수익률은 20일 후 수익률이 가장 높았으므로 20일을 메인으로 제시
        expected_return = {'Avg_Return_Post_20D (%)': '최적 GICS 35.0의 20일 평균 수익률', 'Days_Held': '20'}
        reason = "ALL CONDITIONS MET: Strong Z-Score, Optimal GICS (35.0), High Momentum/Volume."
        
    else:
        decision = 'HOLD'
        # 4가지 필터 중 무엇이 부족했는지 명시
        missing_filters = []
        if not is_surprise_ok: missing_filters.append(f"Z-Score ({Z_SCORE_THRESHOLD})")
        if not is_gics_ok: missing_filters.append(f"GICS Sector ({MOST_SENSITIVE_GICS_SECTORS})")
        if not is_momentum_ok: missing_filters.append(f"Momentum Rate ({MOMENTUM_THRESHOLD*100:.0f}%)")
        if not is_volume_ok: missing_filters.append(f"Volume Ratio ({VOLUME_RATIO_THRESHOLD:.1f}x)")
        
        reason = f"HOLD. Missing Filters: {', '.join(missing_filters)}"
        expected_return = {'Avg_Return_Post_20D (%)': 'N/A', 'Days_Held': '0'}


    # 3. Output Assembly
    return {
        'decision': decision,
        'expected_returns': expected_return,
        'reason': reason
    }

In [3]:
# --- 모델 규칙 저장 (.pkl) ---
model_rules = {
    'model_name': 'Final_Integrated_Classifier',
    'z_threshold': Z_SCORE_THRESHOLD,
    'gics_sectors': MOST_SENSITIVE_GICS_SECTORS, 
    'momentum_threshold': MOMENTUM_THRESHOLD,
    'volume_threshold': VOLUME_RATIO_THRESHOLD,
    'version': 'v3.0 (Z-Score + GICS + Momentum + Volume)'
}

model_path = os.path.join(OUT_DIR, MODEL_FILE_NAME)
with open(model_path, 'wb') as f:
    pickle.dump(model_rules, f)
    
print(f"\n[OK] 최종 모델 Version 3 규칙 저장 완료: {model_path}")


# --- 모델 테스트 (API 호출 시뮬레이션) ---
print("\n" + "="*80)
print("FINAL MODEL (VERSION 3) TEST CASES")
print("="*80)

# Case 1: ALL PASS (Ideal scenario for a BUY)
test_case_1 = get_investment_decision_ver3(
    surprise_z=3.5, 
    gics_code=35.0, 
    momentum_rate=0.10, 
    volume_ratio=3.0
)
print("Case 1 (ALL PASS):", test_case_1['decision'], "| Reason:", test_case_1['reason'])

# Case 2: Momentum FAIL (Expected HOLD)
test_case_2 = get_investment_decision_ver3(
    surprise_z=3.5, 
    gics_code=35.0, 
    momentum_rate=0.01, # FAIL: Below 8%
    volume_ratio=3.0
)
print("Case 2 (MOMENTUM FAIL):", test_case_2['decision'], "| Reason:", test_case_2['reason'])

# Case 3: GICS FAIL (Expected HOLD)
test_case_3 = get_investment_decision_ver3(
    surprise_z=3.5, 
    gics_code=10.0, # FAIL: Wrong GICS
    momentum_rate=0.10, 
    volume_ratio=3.0
)
print("Case 3 (GICS FAIL):", test_case_3['decision'], "| Reason:", test_case_3['reason'])

print("="*80)


[OK] 최종 모델 Version 3 규칙 저장 완료: ../../output/model/model_ver3.pkl

FINAL MODEL (VERSION 3) TEST CASES
Case 1 (ALL PASS): BUY | Reason: ALL CONDITIONS MET: Strong Z-Score, Optimal GICS (35.0), High Momentum/Volume.
Case 2 (MOMENTUM FAIL): HOLD | Reason: HOLD. Missing Filters: Momentum Rate (8%)
Case 3 (GICS FAIL): HOLD | Reason: HOLD. Missing Filters: GICS Sector ([35.0])


In [5]:
import pandas as pd
import numpy as np
import os
import pickle
from typing import Dict, Any

# --- 경로 설정 및 모델 메타데이터 ---
VENDOR_ANALYSIS_PATH = "../../output/problem2_vendor/problem2_vendor_analysis_base.csv"
OUT_DIR = "../../output/model" 
MODEL_FILE_NAME = "model_ver3-1.pkl"
os.makedirs(OUT_DIR, exist_ok=True) 

# --- 모델 하이퍼파라미터 (규칙) 정의 ---
# 1. 서프라이즈 필터 (ARIMA Z-Score)
Z_SCORE_THRESHOLD = 2.0 

# 2. GICS 필터 (최적 산업군: 예시로 35.0 사용)
MOST_SENSITIVE_GICS_SECTORS = [35.0] 

# 3. 모멘텀 필터 (PDF 기반: 20일 기준 8% 이상 상승)
MOMENTUM_THRESHOLD = 0.08 

# 4. 거래량 필터 (PDF 기반: 평소 대비 2배 이상 급증)
VOLUME_RATIO_THRESHOLD = 2.0 

# 분석 및 출력에 사용할 모든 수익률 컬럼
ALL_RETURN_COLUMNS = ['return_post_1d', 'return_post_2d', 'return_post_5d', 'return_post_10d', 'return_post_20d']

In [6]:
def get_investment_decision_ver2(
    surprise_z: float, 
    gics_code: float, 
    momentum_rate: float, 
    volume_ratio: float
) -> str:
    """
    ARIMA Surprise Z-Score, GICS, Momentum, Volume의 4가지 조건을 모두 충족할 때만 BUY 결정.
    """
    
    # SELL signal은 수익성이 실패했으므로, Long-Only 전략으로 진행 (Negative Z-Score는 HOLD로 처리)
    
    # 1. Check all four conditions (ALL-IN)
    is_surprise_ok = (surprise_z > Z_SCORE_THRESHOLD)
    is_gics_ok = (gics_code in MOST_SENSITIVE_GICS_SECTORS)
    is_momentum_ok = (momentum_rate >= MOMENTUM_THRESHOLD)
    is_volume_ok = (volume_ratio >= VOLUME_RATIO_THRESHOLD)
    
    # 2. Final Decision
    if is_surprise_ok and is_gics_ok and is_momentum_ok and is_volume_ok:
        return 'BUY'
    else:
        return 'HOLD'


# --- 3. 모델 실행, 검증 및 저장 ---

print("-> 1. 데이터 로드 및 모델 Version 2 실행...")

try:
    # NOTE: VENDOR_ANALYSIS_PATH 파일에 momentum_rate, volume_ratio, gics_code 컬럼이 포함되어 있다고 가정하고 로드
    df_analysis = pd.read_csv(
        VENDOR_ANALYSIS_PATH, 
        usecols=['symbol', 'date', 'surprise_z', 'gics_code', 'momentum_rate', 'volume_ratio'] + ALL_RETURN_COLUMNS,
        dtype={col: np.float32 for col in ['surprise_z', 'momentum_rate', 'volume_ratio'] + ALL_RETURN_COLUMNS}
    ).dropna(subset=['surprise_z', 'gics_code', 'momentum_rate', 'volume_ratio'])

    # 훈련 데이터의 통계치 (실제 Z-score 계산을 위해 필요)
    # 현재 코드에서는 편의상 Z_SCORE_THRESHOLD를 2.0으로 고정했으나,
    # 실제 API에서는 이 TRAIN_MEAN/STD를 모델에 저장해야 합니다.
    
    # 모든 행에 대해 최종 결정 로직 적용
    df_analysis['decision'] = df_analysis.apply(
        lambda row: get_investment_decision_ver2(
            row['surprise_z'], row['gics_code'], row['momentum_rate'], row['volume_ratio']
        ), axis=1
    )
    
    # 시그널 발생 행만 필터링
    df_signals = df_analysis[df_analysis['decision'] == 'BUY'].copy()
    
    if df_signals.empty:
        print("분석: 최종 통합 필터링 결과, BUY 시그널이 발생하지 않아 검증을 수행할 수 없습니다.")
        exit()

    # 4. 모델 성능 검증 (장기 수익률 요약)
    df_summary = df_signals.groupby('decision')[ALL_RETURN_COLUMNS].mean().mul(100).round(4)
    
    # 컬럼 이름 변경
    column_rename_map = {
        col: f"Avg_Return_Post_{col.replace('return_post_', '').replace('d', 'D')} (%)" 
        for col in ALL_RETURN_COLUMNS
    }
    df_summary = df_summary.rename(columns=column_rename_map)

    # 5. 모델 규칙 저장 (.pkl)
    model_rules = {
        'model_name': 'Integrated_Classifier_V2',
        'z_threshold': Z_SCORE_THRESHOLD,
        'gics_sectors': MOST_SENSITIVE_GICS_SECTORS,
        'momentum_threshold': MOMENTUM_THRESHOLD,
        'volume_threshold': VOLUME_RATIO_THRESHOLD,
        'version': 'v2.0 (4-Factor Integrated)' 
    }
    
    model_path = os.path.join(OUT_DIR, MODEL_FILE_NAME)
    with open(model_path, 'wb') as f:
        pickle.dump(model_rules, f)
        
    print(f"\n[OK] 모델 규칙 저장 완료: {model_path}")
    
    # 6. 결과 출력
    print("\n" + "="*85)
    print("과제 4: 최종 모델 Version 2 장기 수익률 검증 결과 (ALL-IN)")
    print("="*85)
    print(df_summary.to_markdown())
    print("="*85)

except FileNotFoundError as e:
    print(f"❌ 오류: 필요한 파일을 찾을 수 없습니다. (Tech Analysis 컬럼 포함 확인 필요) 경로를 확인하세요: {e}")
except Exception as e:
    print(f"❌ 오류: 데이터 처리 중 예상치 못한 오류 발생: {e}")

-> 1. 데이터 로드 및 모델 Version 2 실행...
❌ 오류: 데이터 처리 중 예상치 못한 오류 발생: Usecols do not match columns, columns expected but not found: ['gics_code', 'momentum_rate', 'volume_ratio']
